## 1. Mount Drive + select data mode

We use **LOCAL mode** by default: datasets live on the Colab
local disk so the active pipeline avoids Drive FUSE small-file
reads. Final artefacts still land on Drive so they survive a
runtime reset.

Override the mode via the ``FALL_DETECTION_DATA_MODE`` env var
(``local`` / ``drive``) or by passing a different root to
``resolve_data_layout``.


In [ ]:
import os, pathlib, sys

# Mount Drive for artefact persistence (always needed).
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not on Colab — Drive mount skipped.")

PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab.data_mode import (
    DataMode,
    resolve_data_layout,
    describe_layout,
)

DATA_MODE = DataMode(os.environ.get("FALL_DETECTION_DATA_MODE", "local").lower())
layout = resolve_data_layout(mode=DATA_MODE)
layout.ensure()

print(f"Data mode        : {layout.mode.value}")
print(f"Dataset root     : {layout.dataset_root}  (active reads happen here)")
print(f"Artefact root    : {layout.artifact_root}  (writes land here)")
print(f"Layout summary   : {describe_layout(layout)}")
if DATA_MODE is DataMode.LOCAL:
    print("Active processing reads from LOCAL disk; only artefacts persist on Drive.")


## 2. Install kagglehub

The Issue 002 dependency (kagglehub) is added to `APPROVED_PIP_PACKAGES`
in Issue 001. We re-run the install here so this notebook works even
if the user opens it before Issue 001's setup notebook.

torch / torchvision are intentionally NOT touched.

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
    check=True,
)
print("kagglehub installed.")

## 3. Set up Kaggle credentials from Colab Secrets

**Before running this cell**, add two secrets via the Colab Secrets UI
(the key-shaped icon on the left sidebar):

  - `KAGGLE_USERNAME` â€” your Kaggle username
  - `KAGGLE_KEY`      â€” your Kaggle API key

The values stay in Colab Secrets. The cell below sets two environment
variables from those secrets and never logs or persists the values.

If you are not on Colab, the staging script will look for a local
`.kaggle/kaggle.json` instead.

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    username = userdata.get("KAGGLE_USERNAME")
    key = userdata.get("KAGGLE_KEY")
    if not username or not key:
        raise RuntimeError(
            "Kaggle credentials missing in Colab Secrets.\n"
            "Add KAGGLE_USERNAME and KAGGLE_KEY in the Secrets panel, "
            "then re-run this cell."
        )
    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = key
    print("Kaggle credentials loaded from Colab Secrets.")
else:
    print("Not on Colab â€” assuming ~/.kaggle/kaggle.json is configured.")

## 4. Stage URFD from Kaggle into Drive

Idempotent: short-circuits when `datasets/urfd/.staged_from_kaggle.txt`
already exists. The marker file proves the tree came from the staging
script (not a stray hand-placed copy).

In [ ]:
from data.stage_urfd import stage_urfd_from_kaggle

staging = stage_urfd_from_kaggle(layout.root)
print(f"URFD staged at: {staging.staged_root}")
print(f"  slug           : {staging.kaggle_slug}")
print(f"  clip folders   : {staging.clip_count}")
print(f"  already staged : {staging.already_staged}")
print()
print("Sample clip folders:")
for clip in staging.clip_folders[:6]:
    print(f"  {clip.folder_name:<24}  label={clip.label:<8}  camera={clip.camera}  seq={clip.clip_sequence}")

## 5. Build the URFD debug manifest

The manifest builder walks the staged tree, parses each folder name
(`fall-NN-camM` â†’ fall, `adl-NN-camM` â†’ no_fall), and emits a
schema-1.1 manifest. The manifest is validated before write so a
broken staged tree can never produce a broken manifest.

In [ ]:
from data.build_urfd_manifest import build_urfd_manifest, write_urfd_manifest

manifest = build_urfd_manifest(staging.staged_root)
manifest_path = staging.staged_root / "manifest.yaml"
write_urfd_manifest(manifest, manifest_path)
print(f"Manifest written: {manifest_path}")
print(f"  schema version : {manifest.schema_version}")
print(f"  clip rows      : {len(manifest.clips)}")
fall_count = sum(1 for c in manifest.clips if c.label.value == "fall")
no_fall_count = sum(1 for c in manifest.clips if c.label.value == "no_fall")
print(f"  fall labels    : {fall_count}")
print(f"  no_fall labels : {no_fall_count}")

## 7. Run YOLO26 + ByteTrack on every URFD clip

**Important — perception reads from local disk, not Drive.**

Live runs showed identical ~1.8 fps on L4 and A100 GPUs. The GPU
was not the bottleneck; Drive FUSE small-file reads were leaving
the GPU idle. We now copy each clip's frames from Drive to
`/content/fall_detection_local/<clip_id>/` BEFORE tracking, then
run the tracker against the local copy. Only the project's artefact
paths on Drive are written back; the raw dataset folder on Drive
is never modified.

Set `SKIP_LOCAL_STAGING=1` in the cell below only if the runtime
already has a faster path to data (e.g. a session that mounted
URFD via a local snapshot). Leave it unset for the standard Colab
GPU runtime.

**Compute policy:** Issue 002 should run on a **T4** with local-disk
frames. The T4 + local-disk combo is sufficient for YOLO26
person detection and ByteTrack tracking at production-relevant FPS.
A100 is **reserved** for compute-bound work, especially VideoMAE
fine-tuning in Issue 006 if it ends up needed; running Issue 002 on
A100 does not help (we just observed ~1.8 fps on both).

For each clip we record:

  - Drive-to-local copy time and throughput (frames/sec).
  - Tracking runtime.
  - Total runtime.
  - FPS after local staging.
  - GPU name + CUDA version + runtime type.

Strict model + tracker checks; no fallback levers applied by default
(baseline is what runs unless proven weak on real footage).
Full-length tracks (`MAX_FRAMES_PER_CLIP=None`) so Issue 003 sees
untruncated clips.


In [ ]:
import time
from perception.local_staging import (
    LocalFrameStager,
    COLAB_LOCAL_ROOT_DEFAULT,
    SKIP_STAGING_ENV_VAR,
)
from perception.frames import FrameFolderReader
from perception.tracker import (
    TrackerConfig,
    run_tracker_on_folder,
    query_gpu_name,
)
from perception.report import build_track_continuity_report
from perception.artifacts import (
    write_perception_artifacts,
    detections_grouped_by_frame,
)
from perception.render import render_annotated_clip
from PIL import Image
import numpy as np
import torch

MAX_FRAMES_PER_CLIP = None  # None = full-length (required by Issue 003); set a small int only for smoke-testing.
ARTIFACTS_ROOT = layout.artifacts / "perception"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

GPU_NAME = query_gpu_name() or 'none (CPU-only runtime)'
RUNTIME_TYPE = 'T4' if GPU_NAME and 'T4' in GPU_NAME else (
    'A100' if GPU_NAME and 'A100' in GPU_NAME else (
    'L4' if GPU_NAME and 'L4' in GPU_NAME else 'unknown GPU'))
print(f"GPU detected     : {GPU_NAME}")
print(f"Runtime type     : {RUNTIME_TYPE}")
print(f"torch CUDA       : {torch.version.cuda}")
print(f"Compute policy   : Issue 002 should run on T4; A100 reserved for compute-bound work (VideoMAE fine-tuning)")
print(f"Local staging    : {COLAB_LOCAL_ROOT_DEFAULT}  (set {SKIP_STAGING_ENV_VAR}=1 to skip)")
print()

config = TrackerConfig()
summary_rows = []
stager = LocalFrameStager(local_root=COLAB_LOCAL_ROOT_DEFAULT)

pipeline_started = time.perf_counter()

for clip in manifest.clips:
    clip_folder = layout.root / clip.source_path
    if not clip_folder.is_dir():
        print(f"[skip] {clip.clip_id}: folder missing on Drive: {clip_folder}")
        continue

    # 1. Discover frames on Drive.
    reader = FrameFolderReader(clip_folder)
    ordered = reader.frames()
    if not ordered:
        print(f"[skip] {clip.clip_id}: no frames in {clip_folder}")
        continue

    capped = ordered if MAX_FRAMES_PER_CLIP is None else ordered[:MAX_FRAMES_PER_CLIP]

    # 2. Stage to local disk (the whole point of this fix).
    with stager.stage_clip(clip.clip_id, clip_folder, capped) as staged:
        local_paths = [Path(p) for p in sorted(staged.local_folder.iterdir())]
        local_frame_count = len(local_paths)
        copy_elapsed = staged.result.elapsed_seconds

        # 3. Track from local disk.
        print(f"[run]  {clip.clip_id}: staged {local_frame_count} frames "
              f"in {copy_elapsed:.1f}s ({staged.result.fps_copy:.0f} frames/s); "
              f"GPU={GPU_NAME}")
        track_started = time.perf_counter()
        run = run_tracker_on_folder(clip.clip_id, local_paths, config=config)
        track_elapsed = time.perf_counter() - track_started
        run.source_folder = str(clip_folder.relative_to(layout.root))

        report = build_track_continuity_report(run, source_folder=run.source_folder)
        out_dir = ARTIFACTS_ROOT / clip.clip_id
        write_perception_artifacts(out_dir, run, report)

        # 4. Render annotated frames (local) for visual review.
        images = [np.array(Image.open(p).convert("RGB")) for p in local_paths]
        detections_by_frame = detections_grouped_by_frame(run.detections, run.frame_count)
        render_annotated_clip(
            images, detections_by_frame,
            output_dir=out_dir / "annotated",
            clip_id=clip.clip_id,
        )

        total_clip = copy_elapsed + track_elapsed
        summary_rows.append({
            "clip_id": clip.clip_id,
            "frames": run.frame_count,
            "detections": run.detection_count,
            "tracks": run.track_count,
            "longest_track_id": report.longest_track_id,
            "longest_track_length": report.longest_track_length,
            "id_switch_count": report.id_switch_count,
            "decode_failures": run.decode_failures,
            "copy_seconds": round(copy_elapsed, 2),
            "copy_fps": round(staged.result.fps_copy, 1),
            "track_seconds": round(track_elapsed, 2),
            "track_fps": round(report.fps, 2),
            "total_clip_seconds": round(total_clip, 2),
        })
        print(f"       -> copy {copy_elapsed:.1f}s | track {track_elapsed:.1f}s | total {total_clip:.1f}s")
        print(f"       -> {report.detection_count} detections, "
              f"{report.track_count} tracks, longest=track {report.longest_track_id} "
              f"({report.longest_track_length} frames), "
              f"{report.id_switch_count} id switches, {report.fps:.1f} fps")
        print(f"       -> artefacts on Drive: {out_dir}")
        # 5. Context manager exit removes the local copy automatically.

pipeline_elapsed = time.perf_counter() - pipeline_started
print()
print(f"Pipeline finished in {pipeline_elapsed:.1f}s.")


## 7. Per-clip summary

The summary rows show what each clip's run produced. Two human-review
questions to ask:

  1. **Did the falling person keep the same track_id through the fall
     window?** Look at `longest_track_id` for the fall clips and
     cross-check the annotated frames in `artifacts/perception/<clip_id>/annotated/`.

  2. **Did ID switches happen at the moment of collapse?** Look at
     `id_switch_count` for fall clips â€” non-zero on a fall clip is
     a known issue per the PRD's *Further Notes* (cheapest killer
     risk: tracking through the fall).

If a fall clip shows zero detections, or if the longest track does
NOT span the fall, **apply the fallback levers** in this order:

  1. Lower `track_low_thresh` (more permissive association).
  2. Switch to BoT-SORT (`fallback_tracker='botsort.yaml'`).
  3. Try `fallback_end2end=False` (BoT-SORT only).

Each lever is exposed on `TrackerConfig`; re-run just this cell with
the chosen fallback set. Record which lever was used in the run meta
JSON the tracker writes.

In [ ]:
import json
from pathlib import Path

summary_path = ARTIFACTS_ROOT / "_run_summary.json"
summary_path.write_text(json.dumps({
    "gpu_name": GPU_NAME,
    "runtime_type": RUNTIME_TYPE,
    "torch_cuda": torch.version.cuda,
    "compute_policy": "Issue 002 should run on T4; A100 reserved for compute-bound work (VideoMAE fine-tuning in Issue 006)",
    "local_staging_root": str(COLAB_LOCAL_ROOT_DEFAULT),
    "max_frames_per_clip": MAX_FRAMES_PER_CLIP,
    "config": {
        "model_name": config.model_name,
        "tracker_config": config.tracker_config,
        "confidence_threshold": config.confidence_threshold,
    },
    "metric_availability": {
        "map_50": "n/a (no detection ground truth)",
        "idf1": "n/a (no tracking ground truth)",
        "mota": "n/a (no tracking ground truth)",
        "hota": "n/a (no tracking ground truth)",
    },
    "pipeline_elapsed_seconds": round(pipeline_elapsed, 2),
    "clips": summary_rows,
}, indent=2), encoding="utf-8")
print(f"Perception summary: {summary_path}")
print()
print(f"  GPU            : {GPU_NAME}")
print(f"  Runtime type   : {RUNTIME_TYPE}")
print(f"  torch CUDA     : {torch.version.cuda}")
print(f"  Total pipeline : {pipeline_elapsed:.1f}s")
print()
print(f"  {'clip_id':<32} {'frames':>7} {'dets':>6} {'tracks':>7} {'longest':>8} {'switches':>9} {'copy_s':>7} {'track_s':>8} {'fps':>6} {'dec_fail':>9}")
for row in summary_rows:
    print(f"  {row['clip_id']:<32} {row['frames']:>7} {row['detections']:>6} "
          f"{row['tracks']:>7} {str(row['longest_track_id']):>8} {row['id_switch_count']:>9} "
          f"{row['copy_seconds']:>7.1f} {row['track_seconds']:>8.1f} {row['track_fps']:>6.1f} {row['decode_failures']:>9}")


## 8. Done

Issue 002 deliverables on Drive:

  - `MyDrive/fall_detection/datasets/urfd/` â€” staged URFD frames.
  - `MyDrive/fall_detection/datasets/urfd/manifest.yaml` â€” URFD debug
    manifest (schema 1.1).
  - `MyDrive/fall_detection/artifacts/perception/<clip_id>/` â€” per clip:
    - `<clip_id>_detections.json` â€” flat detection list.
    - `<clip_id>_tracks.json` â€” per-track summary.
    - `<clip_id>_report.json` â€” track-continuity report.
    - `<clip_id>_run_meta.json` â€” model / tracker / timing metadata.
    - `annotated/<clip_id>_frame_NNNNN.png` â€” boxes + track IDs drawn.
  - `MyDrive/fall_detection/artifacts/perception/_run_summary.json`
    â€” human-readable per-clip summary across the whole run.

Next step readiness:
  - If the falling person keeps the same track_id through the fall
    window on the debug tier â†’ Issue 003 (clip generator) can proceed.
  - If ID switches happen at the moment of collapse on most fall clips â†’
    escalate by trying fallback levers before moving on.